# EDA del listado completo de motos (Revolico)

Fuentes: `data/processed/final/motos_final.json` (538 válidas) y
`data/processed/listados_crudos/motos_listado.json` (2.939 crudas).
Solo estadísticas y distribuciones.

In [1]:
import json
import re
import pandas as pd
from pathlib import Path

BASE = None
for nivel in ("data/processed", "../data/processed", "../../data/processed"):
    if Path(nivel).exists():
        BASE = Path(nivel)
        break
assert BASE is not None

with open(BASE / "final/motos_final.json", encoding="utf-8") as f:
    df_final = pd.DataFrame(json.load(f))
with open(BASE / "listados_crudos/motos_listado.json", encoding="utf-8") as f:
    crudo = json.load(f)
df_crudo = pd.DataFrame(crudo["anuncios"])

print("motos_final.json:", len(df_final))
print("motos_listado.json anuncios:", len(df_crudo))
print("meta crudo:", crudo["total_brutos"], "brutos |", crudo["total_unicos"], "únicos")

motos_final.json: 538
motos_listado.json anuncios: 2939
meta crudo: 3174 brutos | 2939 únicos


In [2]:
MARCAS = [
    ("topmaq", "Topmaq"),
    ("vedca", "Vedca"),
    ("\bizuki\b", "Izuki"),
    ("yoazaky|yoazaki|yoa ?zaky|yoazak", "Yoazaki"),
    ("mountain", "Mountain"),
    ("bucatti|bucati|buccati", "Bucatti"),
    ("fly racing|flywing|\bfly\b", "Fly Racing"),
    ("xmox", "Xmox"),
    ("mishozuki", "Mishozuki"),
    ("\bava\b", "Ava"),
    ("halcon", "Halcon"),
    ("kvitova|kivitova|kitova", "Kvitova"),
    ("murasaki", "Murasaki"),
    ("panda", "Panda"),
    ("\bmaf\b", "Maf"),
    ("\bmaox\b", "Maox"),
    ("\btorvan\b", "Torvan"),
    ("mikazuki", "Mikazuki"),
    ("dcmlz", "DCM-LZ"),
    ("protenzza|protenza|prontenza", "Protenzza"),
    ("voltax", "Voltax"),
    ("volta", "Volta"),
    ("meacon", "Meacon"),
    ("zuzuki|suzuki|\bzuki\b", "Zuki"),
    ("\bsnopy\b", "Snopy"),
    ("yadea", "Yadea"),
    ("\bxtreme\b", "Xtreme"),
    ("mavis", "Mavis"),
    ("\bgt\b", "GT"),
]

def _norm(s):
    plano = re.sub(r"[^a-z0-9]+", " ", (s or "").lower())
    return re.sub(r"\s+", " ", plano).strip()

def marca_de_titulo(titulo):
    texto = _norm(titulo)
    for patron, marca in MARCAS:
        if re.search(patron, texto):
            return marca
    return None

df_crudo["marca"] = df_crudo["titulo"].map(marca_de_titulo)

In [3]:
top15 = df_crudo["marca"].value_counts().head(15)
print("Top 15 marcas en los 2.939 anuncios crudos:")
top15

Top 15 marcas en los 2.939 anuncios crudos:


marca
Topmaq        298
Vedca         128
Yoazaki        90
Mountain       89
Kvitova        77
Bucatti        55
Mishozuki      52
Mikazuki       23
Xmox           18
Fly Racing     17
Volta          17
Protenzza      16
Halcon         15
Panda          12
DCM-LZ          6
Name: count, dtype: int64

In [4]:
import sys
k = Path("src/scrapers") if Path("src/scrapers/revolico.py").exists() else Path("../src/scrapers")
sys.path.insert(0, str(k.resolve()))
import revolico as rv

listados = sorted((rv.CACHE_DIR / "motos").glob("listado_*.html"))
cat = rv.extraer_catalogos(listados[0].read_text(encoding="utf-8"))

muni_map = cat["municipios"]
df_crudo["municipio"] = df_crudo["municipalityId"].map(muni_map)
top_muni = df_crudo["municipio"].value_counts().head(15)
print("Top 15 municipios:")
top_muni

Top 15 municipios:


municipio
Centro Habana            673
Playa                    569
Plaza                    356
Diez de Octubre          237
Cerro                    211
Boyeros                  142
Arroyo Naranjo           122
La Lisa                  113
Marianao                 110
Habana Vieja             108
Cotorro                   61
Habana del Este           58
Guanabacoa                51
San Miguel del Padrón     50
Artemisa                  10
Name: count, dtype: int64

In [5]:
print("Distribución de precios (todos los anuncios, 2.939):")
df_crudo["precio"].describe()
print()
usd = df_crudo[df_crudo["moneda"] == "USD"]
print("Solo moneda USD (%d anuncios):" % len(usd))
usd["precio"].describe()

Distribución de precios (todos los anuncios, 2.939):

Solo moneda USD (2815 anuncios):


count      2815.000000
mean       2309.670334
std        3035.817539
min           1.000000
25%         810.000000
50%        1750.000000
75%        3500.000000
max      111111.000000
Name: precio, dtype: float64

In [6]:
sin_precio = df_crudo["precio"].isna().sum()
print("Anuncios sin precio:", sin_precio)
print("Por moneda:")
df_crudo["moneda"].value_counts(dropna=False)
print()
print("Anuncios (USD) por rango de precio:")
bins = [-1, 500, 1500, 2500, 3500, float("inf")]
etiquetas = ["<500", "500-1500", "1500-2500", "2500-3500", ">3500"]
rango = pd.cut(usd["precio"], bins=bins, labels=etiquetas)
conteo = rango.value_counts().sort_index()
conteo

Anuncios sin precio: 119
Por moneda:

Anuncios (USD) por rango de precio:


precio
<500         302
500-1500     863
1500-2500    629
2500-3500    332
>3500        689
Name: count, dtype: int64